# Phase 1: Persiapan & Ekstraksi Data (Data Wrangling)

Notebook ini berfokus pada pengumpulan data PM2.5 (dari API), memuat data cuaca, dan mengekstrak nilai AOD (Aerosol Optical Depth) dari citra satelit Himawari berdasarkan koordinat geografis stasiun.

## 1.1 Import Library Utama

In [5]:
import os
import urllib.request
import json
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import xarray as xr
import rioxarray

print("All Phase 1 libraries imported successfully!")

All Phase 1 libraries imported successfully!


## 1.2 Pengumpulan Data PM2.5 (Gather PM2.5 Data)

Kita akan mengunduh data PM2.5 historis dari **Open-Meteo Air Quality API** untuk 5 wilayah pemantauan di Jakarta dari 1 Januari 2023 hingga 1 Januari 2026:
1. **Slipi** (Latitude: -6.195, Longitude: 106.8025)
2. **Menteng** (Latitude: -6.1994, Longitude: 106.8392)
3. **Jagakarsa** (Latitude: -6.3341, Longitude: 106.8214)
4. **Jatinegara** (Latitude: -6.2322, Longitude: 106.8811)
5. **Kelapa Gading Indah** (Latitude: -6.1673, Longitude: 106.9052)

Data masing-masing stasiun akan diunduh secara otomatis, disimpan sebagai file CSV individual di folder `data/`, dan digabungkan menjadi satu file utama `data/combined_pm25.csv`.

In [8]:
import os
import urllib.request
import json
import pandas as pd

# Daftar stasiun dan URL API masing-masing (Koordinat Slipi disesuaikan ke -6.195, 106.8025)
stations = {
    'Slipi': 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-6.195&longitude=106.8025&hourly=pm2_5&start_date=2023-01-01&end_date=2026-01-01',
    'Menteng': 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-6.1994&longitude=106.8392&hourly=pm2_5&start_date=2023-01-01&end_date=2026-01-01',
    'Jagakarsa': 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-6.3341&longitude=106.8214&hourly=pm2_5&start_date=2023-01-01&end_date=2026-01-01',
    'Jatinegara': 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-6.2322&longitude=106.8811&hourly=pm2_5&start_date=2023-01-01&end_date=2026-01-01',
    'Kelapa Gading Indah': 'https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-6.1673&longitude=106.9052&hourly=pm2_5&start_date=2023-01-01&end_date=2026-01-01'
}

# Buat folder 'data/pm25' jika belum ada
pm25_dir = os.path.join('data', 'pm25')
os.makedirs(pm25_dir, exist_ok=True)

all_dfs = []

for name, url in stations.items():
    try:
        print(f"Mengunduh data PM2.5 untuk stasiun {name}...")
        with urllib.request.urlopen(url) as response:
            raw_data = json.loads(response.read().decode())

        # Ekstrak data hourly
        hourly_data = raw_data['hourly']
        df_station = pd.DataFrame(hourly_data)

        # Ubah nama kolom untuk konsistensi
        df_station.rename(columns={'time': 'waktu', 'pm2_5': 'PM2.5'}, inplace=True)

        # Tambahkan metadata
        df_station['stasiun'] = name
        df_station['latitude'] = raw_data['latitude']
        df_station['longitude'] = raw_data['longitude']
        df_station['waktu'] = pd.to_datetime(df_station['waktu'])

        # Simpan file individual ke folder 'data/pm25'
        sanitized_name = name.lower().replace(" ", "_")
        individual_path = os.path.join(pm25_dir, f'{sanitized_name}_pm25.csv')
        df_station.to_csv(individual_path, index=False)
        print(f"-> Berhasil menyimpan data stasiun {name} ke: {individual_path}")

        # Masukkan ke list utama
        all_dfs.append(df_station)

    except Exception as e:
        print(f"-> Gagal mengunduh data {name}: {e}")

# Gabungkan seluruh stasiun menjadi satu dataset jika ada yang berhasil diunduh
if all_dfs:
    df_combined = pd.concat(all_dfs, ignore_index=True)
    combined_path = os.path.join(pm25_dir, 'combined_pm25.csv')
    df_combined.to_csv(combined_path, index=False)
    print("\n" + "="*60)
    print(f"Berhasil menggabungkan data seluruh stasiun!")
    print(f"File gabungan disimpan di: {combined_path}")
    print(f"Total baris data gabungan: {len(df_combined)}")
    print(f"Stasiun yang terdaftar: {list(df_combined['stasiun'].unique())}")
    print("="*60)
else:
    print("Tidak ada data stasiun yang berhasil diunduh.")

Mengunduh data PM2.5 untuk stasiun Slipi...
-> Berhasil menyimpan data stasiun Slipi ke: data\pm25\slipi_pm25.csv
Mengunduh data PM2.5 untuk stasiun Menteng...
-> Berhasil menyimpan data stasiun Menteng ke: data\pm25\menteng_pm25.csv
Mengunduh data PM2.5 untuk stasiun Jagakarsa...
-> Berhasil menyimpan data stasiun Jagakarsa ke: data\pm25\jagakarsa_pm25.csv
Mengunduh data PM2.5 untuk stasiun Jatinegara...
-> Berhasil menyimpan data stasiun Jatinegara ke: data\pm25\jatinegara_pm25.csv
Mengunduh data PM2.5 untuk stasiun Kelapa Gading Indah...
-> Berhasil menyimpan data stasiun Kelapa Gading Indah ke: data\pm25\kelapa_gading_indah_pm25.csv

Berhasil menggabungkan data seluruh stasiun!
File gabungan disimpan di: data\pm25\combined_pm25.csv
Total baris data gabungan: 131640
Stasiun yang terdaftar: ['Slipi', 'Menteng', 'Jagakarsa', 'Jatinegara', 'Kelapa Gading Indah']


## 1.3 Pengumpulan Data Cuaca (Gather Weather Data)

Fitur cuaca yang diambil meliputi:
- `temperature_2m` (Suhu udara)
- `apparent_temperature` (Suhu semu)
- `relative_humidity_2m` (Kelembapan relatif)
- `dew_point_2m` (Titik embun)
- `precipitation` (Curah hujan)
- `rain` (Hujan)
- `surface_pressure` (Tekanan permukaan)
- `cloud_cover` (Total tutupan awan)
- `wind_speed_10m` (Kecepatan angin)
- `wind_direction_10m` (Arah angin)

Data cuaca ini akan diunduh, disimpan ke folder **`data/cuaca/`**, kemudian digabungkan secara langsung (*merge*) dengan data PM2.5. Hasil akhirnya disimpan ke file **`data/raw_merged_data.csv`**.

In [9]:
import os
import urllib.request
import json
import pandas as pd

# Daftar koordinat stasiun yang disesuaikan
station_locations = {
    'Slipi': {'lat': -6.195, 'lon': 106.8025},
    'Menteng': {'lat': -6.1994, 'lon': 106.8392},
    'Jagakarsa': {'lat': -6.3341, 'lon': 106.8214},
    'Jatinegara': {'lat': -6.2322, 'lon': 106.8811},
    'Kelapa Gading Indah': {'lat': -6.1673, 'lon': 106.9052}
}

start_date = "2023-01-01"
end_date = "2026-01-01"

# Buat folder 'data/cuaca' jika belum ada
cuaca_dir = os.path.join('data', 'cuaca')
os.makedirs(cuaca_dir, exist_ok=True)

weather_dfs = []

for name, coords in station_locations.items():
    lat, lon = coords['lat'], coords['lon']

    # Menggunakan archive-api untuk rentang waktu historis 2023 s.d. 2026
    url_weather = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,rain,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m"

    try:
        print(f"Mengunduh data cuaca untuk stasiun {name}...")
        with urllib.request.urlopen(url_weather) as response:
            raw_data = json.loads(response.read().decode())

        hourly_data = raw_data['hourly']
        df_w = pd.DataFrame(hourly_data)

        # Sesuaikan format nama kolom dan waktu
        df_w.rename(columns={'time': 'waktu', 'cloud_cover': 'cloud_cover_total'}, inplace=True)
        df_w['stasiun'] = name
        df_w['latitude'] = raw_data['latitude']
        df_w['longitude'] = raw_data['longitude']
        df_w['waktu'] = pd.to_datetime(df_w['waktu'])

        # Simpan file cuaca individual ke folder 'data/cuaca'
        sanitized_name = name.lower().replace(" ", "_")
        individual_weather_path = os.path.join(cuaca_dir, f'{sanitized_name}_weather.csv')
        df_w.to_csv(individual_weather_path, index=False)
        print(f"-> Berhasil menyimpan data cuaca stasiun {name} ke: {individual_weather_path}")

        weather_dfs.append(df_w)

    except Exception as e:
        print(f"-> Gagal mengunduh data cuaca {name}: {e}")

# Gabungkan seluruh stasiun menjadi satu dataset cuaca jika ada yang berhasil diunduh
if weather_dfs:
    df_weather_combined = pd.concat(weather_dfs, ignore_index=True)
    combined_weather_path = os.path.join(cuaca_dir, 'combined_weather.csv')
    df_weather_combined.to_csv(combined_weather_path, index=False)
    print("\n" + "="*60)
    print("PENGUNDUHAN DATA CUACA SELESAI!")
    print(f"File gabungan cuaca disimpan di: {combined_weather_path}")
    print(f"Total baris data cuaca gabungan: {len(df_weather_combined)}")
    print("Kolom yang tersedia:")
    for col in df_weather_combined.columns:
        print(f"- {col}")
    print("="*60)
else:
    print("Tidak ada data cuaca yang berhasil diunduh.")

Mengunduh data cuaca untuk stasiun Slipi...
-> Berhasil menyimpan data cuaca stasiun Slipi ke: data\cuaca\slipi_weather.csv
Mengunduh data cuaca untuk stasiun Menteng...
-> Berhasil menyimpan data cuaca stasiun Menteng ke: data\cuaca\menteng_weather.csv
Mengunduh data cuaca untuk stasiun Jagakarsa...
-> Berhasil menyimpan data cuaca stasiun Jagakarsa ke: data\cuaca\jagakarsa_weather.csv
Mengunduh data cuaca untuk stasiun Jatinegara...
-> Berhasil menyimpan data cuaca stasiun Jatinegara ke: data\cuaca\jatinegara_weather.csv
Mengunduh data cuaca untuk stasiun Kelapa Gading Indah...
-> Berhasil menyimpan data cuaca stasiun Kelapa Gading Indah ke: data\cuaca\kelapa_gading_indah_weather.csv

PENGUNDUHAN DATA CUACA SELESAI!
File gabungan cuaca disimpan di: data\cuaca\combined_weather.csv
Total baris data cuaca gabungan: 131640
Kolom yang tersedia:
- waktu
- temperature_2m
- apparent_temperature
- relative_humidity_2m
- dew_point_2m
- precipitation
- rain
- surface_pressure
- cloud_cover_tota

## 1.4 Pengumpulan Data AOD Himawari (Gather Himawari AOD)

Pada tahap ini, kita akan mengekstrak nilai **AOD (Aerosol Optical Depth)** dari citra satelit Himawari-8/9 (biasanya berformat NetCDF `.nc` atau GeoTIFF `.tif`) berdasarkan koordinat asli stasiun dan waktu yang bersesuaian.

### Lihat file struktur ftp

In [ ]:
import ftplib
import os

CONF = {
    "host": "ftp.ptree.jaxa.jp",
    "user": "qois.firosi_gmail.com",
    "pass": "SP+wari8",
    "ver": "031"
}

try:
    print("Menghubungkan ke FTP...")
    ftp = ftplib.FTP(CONF['host'])
    ftp.login(user=CONF['user'], passwd=CONF['pass'])
    ftp.set_pasv(True)

    # 1. Intip folder jam 00 pada 1 Januari 2023 (awal data)
    path_2023 = f"/pub/himawari/L2/ARP/{CONF['ver']}/202301/01/00"
    print(f"\n[1] Membuka folder: {path_2023}")
    try:
        ftp.cwd(path_2023)
        files_2023 = []
        ftp.dir(files_2023.append)
        print("Daftar file di folder jam 00 (1 Jan 2023):")
        for f in files_2023[:5]:
            print("  ", f)
    except Exception as e_2023:
        print(f"✕ Gagal membuka folder 2023: {e_2023}")

    # 2. Intip folder jam 00 pada 1 Januari 2024 (tengah data)
    path_2024 = f"/pub/himawari/L2/ARP/{CONF['ver']}/202401/01/00"
    print(f"\n[2] Membuka folder: {path_2024}")
    try:
        ftp.cwd(path_2024)
        files_2024 = []
        ftp.dir(files_2024.append)
        print("Daftar file di folder jam 00 (1 Jan 2024):")
        for f in files_2024[:5]:
            print("  ", f)
    except Exception as e_2024:
        print(f"✕ Gagal membuka folder 2024: {e_2024}")

    ftp.quit()
    print("\n✓ Koneksi FTP ditutup.")

except Exception as e:
    print(f"\n✕ Gagal terhubung: {e}")

Menghubungkan ke FTP...

[1] Membuka folder: /pub/himawari/L2/ARP/031/202301/01/00
Daftar file di folder jam 00 (1 Jan 2023):
   -rw-r--r--   1 2100     210       4997488 Jan  1  2023 NC_H09_20230101_0000_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   1 2100     210       5040257 Jan  1  2023 NC_H09_20230101_0010_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   1 2100     210       5022769 Jan  1  2023 NC_H09_20230101_0020_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   1 2100     210       5006570 Jan  1  2023 NC_H09_20230101_0030_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   1 2100     210       5016516 Jan  1  2023 NC_H09_20230101_0040_L2ARP031_FLDK.02401_02401.nc

[2] Membuka folder: /pub/himawari/L2/ARP/031/202401/01/00
Daftar file di folder jam 00 (1 Jan 2024):
   -rw-r--r--   1 2100     210       6494105 Jan  1  2024 NC_H09_20240101_0000_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   1 2100     210       6542517 Jan  1  2024 NC_H09_20240101_0010_L2ARP031_FLDK.02401_02401.nc
   -rw-r--r--   

### Test ambil data aod / aot di saat memang tidak ada cloud cover

In [ ]:
import ftplib
import os
import xarray as xr
import numpy as np

CONF = {
    "host": "ftp.ptree.jaxa.jp",
    "user": "qois.firosi_gmail.com",
    "pass": "SP+wari8"
}

target_lat = -6.1950
target_lon = 106.8025
test_nc = "inspect_dry_season.nc"

try:
    print("Menghubungkan ke FTP untuk mengambil data musim kemarau...")
    ftp = ftplib.FTP(CONF['host'])
    ftp.login(user=CONF['user'], passwd=CONF['pass'])
    ftp.set_pasv(True)

    # Navigasi ke folder tanggal 15 Agustus 2023 jam 04 UTC (11:00 WIB)
    ftp.cwd("/pub/himawari/L2/ARP/031/202308/15/04")

    # Cari nama file yang cocok
    files_list = []
    ftp.dir(files_list.append)
    target_file = None
    for line in files_list:
        fname = line.split()[-1]
        if "_20230815_0400_" in fname and fname.endswith('.nc'):
            target_file = fname
            break

    if not target_file:
        raise FileNotFoundError("File target untuk 15 Agt 2023 jam 04:00 UTC tidak ditemukan.")

    print(f"Mengunduh file {target_file}...")
    ftp.retrbinary(f"RETR {target_file}", open(test_nc, 'wb').write)
    ftp.quit()
    print("✓ Berhasil terdownload.")

    with xr.open_dataset(test_nc) as ds:
        aot_var = ds['AOT']

        # 1. Ambil nilai di Slipi terdekat
        val_slipi = aot_var.sel(latitude=target_lat, longitude=target_lon, method='nearest').values.item()
        print(f"\n✓ Nilai AOT di Slipi (-6.1950, 106.8025): {val_slipi}")

        # 2. Cek validitas piksel di seluruh area Jakarta Box
        jakarta_slice = aot_var.sel(latitude=slice(-6.10, -6.35), longitude=slice(106.70, 106.95))
        valid_pixels = np.count_nonzero(~np.isnan(jakarta_slice.values))
        total_pixels = jakarta_slice.values.size
        print(f"✓ Total piksel di Jakarta Box: {total_pixels}")
        print(f"✓ Piksel valid (Bukan NaN): {valid_pixels} ({valid_pixels/total_pixels*100:.2f}%)")

        if valid_pixels > 0:
            print("✓ Sampel nilai AOT valid di Jakarta:", jakarta_slice.values[~np.isnan(jakarta_slice.values)][:5])

    if os.path.exists(test_nc):
        os.remove(test_nc)

except Exception as e:
    print(f"✕ Gagal: {e}")
    if os.path.exists(test_nc):
        os.remove(test_nc)

Menghubungkan ke FTP untuk mengambil data musim kemarau...
Mengunduh file NC_H09_20230815_0400_L2ARP031_FLDK.02401_02401.nc...
✓ Berhasil terdownload.

✓ Nilai AOT di Slipi (-6.1950, 106.8025): nan
✓ Total piksel di Jakarta Box: 30
✓ Piksel valid (Bukan NaN): 9 (30.00%)
✓ Sampel nilai AOT valid di Jakarta: [0.7326 0.6346 0.7188 0.8854 0.681 ]


### Opsi 1 untuk fix jika ada cloud cover: Pengambilan Radius Terdekat (Spatial Buffer / Nearest Valid)

In [ ]:
import ftplib
import os
import xarray as xr
import numpy as np

CONF = {
    "host": "ftp.ptree.jaxa.jp",
    "user": "qois.firosi_gmail.com",
    "pass": "SP+wari8"
}

target_lat = -6.1950
target_lon = 106.8025
test_nc = "inspect_rainy_season.nc"

try:
    print("Menghubungkan ke FTP untuk mengambil data musim hujan (1 Jan 2023)...")
    ftp = ftplib.FTP(CONF['host'])
    ftp.login(user=CONF['user'], passwd=CONF['pass'])
    ftp.set_pasv(True)

    # Menggunakan file musim hujan 1 Jan 2023 pukul 00:00 UTC (07:00 WIB)
    ftp.cwd("/pub/himawari/L2/ARP/031/202301/01/00")
    filename = "NC_H09_20230101_0000_L2ARP031_FLDK.02401_02401.nc"

    print(f"Mengunduh file {filename}...")
    ftp.retrbinary(f"RETR {filename}", open(test_nc, 'wb').write)
    ftp.quit()
    print("✓ Berhasil terdownload.")

    print("\nMencari piksel AOT valid terdekat dari Slipi...")
    with xr.open_dataset(test_nc) as ds:
        aot_var = ds['AOT']
        aot_values = aot_var.values

        # Buat mask untuk piksel yang tidak NaN
        valid_mask = ~np.isnan(aot_values)

        # Dapatkan indeks baris (latitude) dan kolom (longitude) dari piksel yang valid
        lat_indices, lon_indices = np.where(valid_mask)

        if len(lat_indices) > 0:
            # Ambil nilai koordinat nyata dari indeks tersebut
            valid_lats = ds['latitude'].values[lat_indices]
            valid_lons = ds['longitude'].values[lon_indices]
            valid_aot = aot_values[valid_mask]

            # Hitung jarak kuadrat (Euclidean) dari koordinat Slipi ke semua koordinat valid
            distances_sq = (valid_lats - target_lat)**2 + (valid_lons - target_lon)**2

            # Cari indeks dengan jarak terkecil
            min_idx = np.argmin(distances_sq)

            nearest_lat = valid_lats[min_idx]
            nearest_lon = valid_lons[min_idx]
            nearest_val = valid_aot[min_idx]
            distance_deg = np.sqrt(distances_sq[min_idx])

            # Konversi derajat ke kilometer (1 derajat ~ 111.32 km)
            distance_km = distance_deg * 111.32

            print("\n" + "="*60)
            print("HASIL PENGUJIAN OPSI 1 PADA MUSIM HUJAN (1 JAN 2023):")
            print(f"Stasiun Asal: Slipi ({target_lat}, {target_lon})")
            print(f"Piksel Valid Terdekat Ditemukan di: ({nearest_lat:.4f}, {nearest_lon:.4f})")
            print(f"Jarak Spasial: {distance_deg:.4f} derajat (~ {distance_km:.2f} km)")
            print(f"Nilai AOT Terdekat: {nearest_val:.4f}")
            print("="*60)

            # Evaluasi kelayakan
            if distance_km < 10:
                print("\n[Hasil] Jarak dekat (< 10 km). Teori Opsi 1 LAYAK digunakan di musim hujan.")
            elif distance_km < 25:
                print("\n[Hasil] Jarak sedang (10 - 25 km). Cukup layak, namun representasi kualitas udara lokal berkurang.")
            else:
                print("\n[Hasil] Jarak terlalu jauh (> 25 km). Opsi 1 TIDAK LAYAK untuk jam ini.")
                print("Artinya, seluruh Jabodetabek/Jawa bagian barat tertutup awan tebal.")
        else:
            print("\n[!] Mengejutkan: Tidak ada satupun piksel valid di seluruh citra Full Disk satelit!")

    if os.path.exists(test_nc):
        os.remove(test_nc)

except Exception as e:
    print(f"✕ Gagal: {e}")
    if os.path.exists(test_nc):
        os.remove(test_nc)

Menghubungkan ke FTP untuk mengambil data musim hujan (1 Jan 2023)...
Mengunduh file NC_H09_20230101_0000_L2ARP031_FLDK.02401_02401.nc...
✓ Berhasil terdownload.

Mencari piksel AOT valid terdekat dari Slipi...

HASIL PENGUJIAN OPSI 1 PADA MUSIM HUJAN (1 JAN 2023):
Stasiun Asal: Slipi (-6.195, 106.8025)
Piksel Valid Terdekat Ditemukan di: (-7.8500, 109.9500)
Jarak Spasial: 3.5561 derajat (~ 395.86 km)
Nilai AOT Terdekat: 1.5298

[Hasil] Jarak terlalu jauh (> 25 km). Opsi 1 TIDAK LAYAK untuk jam ini.
Artinya, seluruh Jabodetabek/Jawa bagian barat tertutup awan tebal.


*Jarak 395.86 km (piksel valid terdekat ada di Jawa Tengah/Yogyakarta)* membuktikan secara empiris bahwa Opsi 1 (Radius Terdekat) sama sekali tidak layak digunakan pada musim hujan karena tutupan awan monsun bersifat regional (menutupi seluruh Jawa bagian barat).

Dengan gugurnya Opsi 1 untuk kondisi mendung total, berikut adalah evaluasi dan langkah praktis terbaik untuk melangkah ke depan: 

🗺️ Rencana (Plan) Langkah ke Depan

#### Langkah 1: Selesaikan Pengumpulan Data AOD (Fase 1)
tetap mengunduh data AOT dari FTP JAXA apa adanya (termasuk nilai NaN saat mendung). Output disimpan di data/aod/combined_aod.csv.

#### Langkah 2: Pembersihan & Penanganan NaN untuk RFR (Fase 2 - Notebook 02)
Karena Random Forest bawaan scikit-learn membutuhkan data tanpa NaN, kita akan menerapkan teknik Imputasi Nilai Sentinel (Sentinel Value Imputation) di Notebook 02:

- Tekniknya: Kita ubah semua nilai NaN pada AOT menjadi nilai penanda khusus, misalnya -1.0 atau -999.0.
- Mengapa ini bekerja di RFR? Random Forest didasarkan pada pohon keputusan (decision trees). Pohon keputusan akan belajar membuat pembagian (split) otomatis:
- Cabang kiri: Jika AOT<0 (berarti mendung/malam), gunakan fitur cuaca untuk memprediksi PM2.5.
- Cabang kanan: Jika AOT≥0 (berarti cerah), gunakan gabungan fitur cuaca dan nilai AOT riil.
- Hasil: RFR tetap dapat berjalan dengan lancar, memanfaatkan data cuaca saat mendung, dan memanfaatkan data AOD satelit saat langit cerah.

#### Langkah 3: Analisis Deskriptif & Rekayasa Fitur (Fase 3 - Notebook 03)
melakukan EDA untuk melihat korelasi PM2.5 dengan AOT (pada jam cerah) serta korelasi PM2.5 dengan cuaca (suhu, kelembapan, angin).
Membuat fitur tambahan seperti waktu (jam, hari, bulan) dan spasial (koordinat asli stasiun).
Menggabungkan ketiganya (PM2.5, Cuaca, AOT) menjadi satu file siap latih: data/model_ready_data.csv.

#### Langkah 4: Pelatihan Model RFR (Fase 4 - Notebook 04)
Melatih RFR menggunakan dataset akhir.
Menerapkan validasi LOSOCV (Leave-One-Station-Out) untuk membuktikan keandalan model di lokasi baru tanpa stasiun.

### 1.4 Pengunduhan Data AOD Himawari dari JAXA FTP (Daytime Only)

Pada tahap ini, kita akan mengunduh data **AOT (Aerosol Optical Thickness)** dari satelit Himawari-9 Level 2 ARP versi `031` secara otomatis menggunakan koneksi FTP ke server JAXA P-Tree.

#### ⚙️ Alur Kerja & Optimalisasi Komputasi:
1. **Filter Jam Siang (Daytime)**: Satelit hanya mendeteksi AOD pada siang hari (memanfaatkan pantulan matahari), yaitu pukul **00:00 s.d. 09:00 UTC** (07:00 s.d. 16:00 WIB). Program hanya mengunduh data pada jam-jam tersebut.
2. **Auto-Delete (Hemat Penyimpanan)**: File NetCDF (.nc) berukuran besar (~5 MB per file) akan diunduh secara sementara ke komputer lokal, diekstrak nilai koordinatnya untuk kelima stasiun, lalu langsung dihapus otomatis. Ruang harddisk yang dibutuhkan hanya sekitar 10-20 MB.
3. **Auto-Resume**: Jika proses terputus, Anda dapat menjalankan ulang cell ini dan program akan melanjutkan sisa data yang belum diunduh berdasarkan file `data/aod/combined_aod.csv` yang sudah terbentuk.
4. **Koordinat Stasiun Asli**:
   * **Slipi**: Lat `-6.1950`, Lon `106.8025`
   * **Menteng**: Lat `-6.1994`, Lon `106.8392`
   * **Jagakarsa**: Lat `-6.3341`, Lon `106.8214`
   * **Jatinegara**: Lat `-6.2322`, Lon `106.8811`
   * **Kelapa Gading Indah**: Lat `-6.1673`, Lon `106.9052`

*Catatan: Pastikan mengubah variabel `TEST_MODE = False` di cell berikutnya untuk mengunduh seluruh data historis dari tahun 2023 s.d. 2026 secara penuh.*

In [25]:
import ftplib
import os
import json
import urllib.request
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

# ==========================================
# 1. KONFIGURASI UTAMA
# ==========================================
CONF = {
    "host": "ftp.ptree.jaxa.jp",
    "user": "qois.firosi_gmail.com",
    "pass": "SP+wari8",
    "ver": "031"
}

# Lokasi stasiun asli (High Resolution untuk citra satelit)
station_coords = {
    'Slipi': {'lat': -6.1950, 'lon': 106.8025},
    'Menteng': {'lat': -6.1994, 'lon': 106.8392},
    'Jagakarsa': {'lat': -6.3341, 'lon': 106.8214},
    'Jatinegara': {'lat': -6.2322, 'lon': 106.8811},
    'Kelapa Gading Indah': {'lat': -6.1673, 'lon': 106.9052}
}

# Tentukan folder output & file output
aod_dir = os.path.join('data', 'aod')
os.makedirs(aod_dir, exist_ok=True)
output_file = os.path.join(aod_dir, 'combined_aod.csv')

# UJI COBA DAHSYAT:
# - Ubah ke True jika ingin menguji coba 5 file pertama saja.
# - Ubah ke False jika sudah siap mendownload data penuh (2023 s.d 2026).
TEST_MODE = False

# ==========================================
# 2. LOAD TIMESTAMPS DARI PM2.5 CSV
# ==========================================
combined_pm25_path = os.path.join('data', 'pm25', 'combined_pm25.csv')
if not os.path.exists(combined_pm25_path):
    raise FileNotFoundError(f"File {combined_pm25_path} tidak ditemukan! Silakan jalankan cell PM2.5 terlebih dahulu.")

# Baca waktu unik dari data PM2.5
df_pm25 = pd.read_csv(combined_pm25_path)
df_pm25['waktu'] = pd.to_datetime(df_pm25['waktu'])
unique_times_wib = sorted(df_pm25['waktu'].unique())

print(f"Total rentang waktu terdeteksi (WIB): {len(unique_times_wib)} jam.")

# ==========================================
# 3. FILTER DAN AUTO-RESUME
# ==========================================
completed_records = set()
if os.path.exists(output_file):
    try:
        df_existing = pd.read_csv(output_file)
        df_existing['waktu'] = pd.to_datetime(df_existing['waktu'])
        # Catat waktu + stasiun yang sudah sukses diekstrak
        for _, row in df_existing.dropna(subset=['AOD']).iterrows():
            completed_records.add((row['waktu'], row['stasiun']))
        print(f"Ditemukan data lama. {len(completed_records)} record stasiun telah ter-ekstrak sebelumnya.")
    except Exception as e:
        print(f"Gagal memuat file lama (akan menulis ulang): {e}")

# Kumpulkan daftar task unduhan (hanya jam siang: UTC 00:00 s.d 09:00 / WIB 07:00 s.d 16:00)
tasks = []
for t_wib in unique_times_wib:
    t_wib = pd.Timestamp(t_wib)
    t_utc = t_wib - pd.Timedelta(hours=7)

    # Himawari AOD hanya aktif pada siang hari (UTC 00:00 s.d 09:00)
    if 0 <= t_utc.hour <= 9:
        # Periksa apakah stasiun untuk waktu ini belum selesai diekstrak
        stations_to_process = [st for st in station_coords.keys() if (t_wib, st) not in completed_records]
        if stations_to_process:
            tasks.append({
                'waktu_wib': t_wib,
                'waktu_utc': t_utc,
                'stations': stations_to_process
            })

print(f"Jumlah jam siang yang harus diproses/diunduh: {len(tasks)} jam.")

if TEST_MODE:
    tasks = tasks[:5]
    print(f"[TEST MODE AKTIF] Hanya memproses {len(tasks)} jam sampel pertama.")

# ==========================================
# 4. DOWNLOAD DAN EKSTRAKSI LOOP
# ==========================================
if len(tasks) > 0:
    try:
        print("Menghubungkan ke JAXA FTP...")
        ftp = ftplib.FTP(CONF['host'])
        ftp.login(user=CONF['user'], passwd=CONF['pass'])
        ftp.set_pasv(True)
        print("✓ Terhubung ke FTP.")

        extracted_data = []
        temp_nc = "temp_aod.nc"  # Tempat penyimpanan sementara

        for idx, task in enumerate(tasks):
            t_utc = task['waktu_utc']
            t_wib = task['waktu_wib']

            # Format nama folder FTP: /pub/himawari/L2/ARP/031/YYYYMM/DD/HH/
            ym = t_utc.strftime("%Y%m")
            dd = t_utc.strftime("%d")
            hh = t_utc.strftime("%H")

            ftp_dir = f"/pub/himawari/L2/ARP/{CONF['ver']}/{ym}/{dd}/{hh}"

            # Pola pencarian nama file menit '00' pada jam tersebut (misal: "_20230101_0000_")
            time_pattern = f"_{t_utc.strftime('%Y%m%d_%H00')}_"

            print(f"[{idx+1}/{len(tasks)}] Memproses {t_wib} WIB...")

            try:
                # Navigasi ke folder FTP jam tersebut
                ftp.cwd(ftp_dir)

                # Baca daftar file di folder untuk mencari file yang cocok dengan pola
                file_list = []
                ftp.dir(file_list.append)

                target_filename = None
                for line in file_list:
                    parts = line.split()
                    if len(parts) > 0:
                        fname = parts[-1]
                        if time_pattern in fname and fname.endswith('.nc'):
                            target_filename = fname
                            break

                if target_filename is None:
                    raise FileNotFoundError(f"Tidak ada file dengan pola {time_pattern} di direktori ini.")

                # Download file NetCDF secara lokal
                print(f"  -> Mengunduh: {target_filename}...")
                with open(temp_nc, 'wb') as f_local:
                    ftp.retrbinary(f"RETR {target_filename}", f_local.write)

                # Ekstrak data menggunakan xarray
                with xr.open_dataset(temp_nc) as ds:
                    for st in task['stations']:
                        lat = station_coords[st]['lat']
                        lon = station_coords[st]['lon']

                        try:
                            # Menggunakan 'AOT' (nama variabel AOD asli di file JAXA)
                            aod_val = ds['AOT'].sel(latitude=lat, longitude=lon, method='nearest').values
                            if isinstance(aod_val, np.ndarray):
                                aod_val = float(aod_val.item())

                            # Filter nilai kosong (out of bounds / invalid)
                            if aod_val < 0 or aod_val > 5 or np.isnan(aod_val):
                                aod_val = np.nan

                        except Exception:
                            aod_val = np.nan

                        extracted_data.append({
                            'waktu': t_wib,
                            'stasiun': st,
                            'AOD': aod_val  # Disimpan dengan nama kolom AOD
                        })

                # Hapus file NetCDF lokal setelah dibaca
                if os.path.exists(temp_nc):
                    os.remove(temp_nc)

            except Exception as e_download:
                print(f"  -> File gagal diperoleh/diproses: {e_download}")
                if os.path.exists(temp_nc):
                    os.remove(temp_nc)
                # Isi nilai NaN untuk kegagalan
                for st in task['stations']:
                    extracted_data.append({
                        'waktu': t_wib,
                        'stasiun': st,
                        'AOD': np.nan
                    })

            # Simpan bertahap setiap 20 file agar aman jika terjadi crash
            if (idx + 1) % 20 == 0 or (idx + 1) == len(tasks):
                df_new = pd.DataFrame(extracted_data)

                if os.path.exists(output_file):
                    df_old = pd.read_csv(output_file)
                    df_final = pd.concat([df_old, df_new]).drop_duplicates(subset=['waktu', 'stasiun'], keep='last')
                else:
                    df_final = df_new

                df_final.to_csv(output_file, index=False)
                extracted_data = []  # Reset penampung data baru
                print(f"  ✓ Progres berhasil disimpan ke: {output_file}")

        ftp.quit()
        print("\n✓ Pengunduhan dan ekstraksi selesai.")

    except Exception as e_ftp:
        print(f"\n✕ Gagal melakukan penelusuran FTP: {e_ftp}")
else:
    print("Semua data jam siang sudah selesai diekstrak sebelumnya.")

# Tampilkan hasil data AOD yang berhasil disimpan
if os.path.exists(output_file):
    df_result = pd.read_csv(output_file)
    print("\n" + "="*50)
    print(f"DATA AOD BERHASIL DISIMPAN!")
    print(f"Lokasi file: {output_file}")
    print(f"Total baris data: {len(df_result)}")
    print(df_result.dropna(subset=['AOD']).head())
    print("="*50)

Total rentang waktu terdeteksi (WIB): 26328 jam.
Ditemukan data lama. 0 record stasiun telah ter-ekstrak sebelumnya.
Jumlah jam siang yang harus diproses/diunduh: 10970 jam.
Menghubungkan ke JAXA FTP...
✓ Terhubung ke FTP.
[1/10970] Memproses 2023-01-01 07:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0000_L2ARP031_FLDK.02401_02401.nc...
[2/10970] Memproses 2023-01-01 08:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0100_L2ARP031_FLDK.02401_02401.nc...
[3/10970] Memproses 2023-01-01 09:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0200_L2ARP031_FLDK.02401_02401.nc...
[4/10970] Memproses 2023-01-01 10:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0300_L2ARP031_FLDK.02401_02401.nc...
[5/10970] Memproses 2023-01-01 11:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0400_L2ARP031_FLDK.02401_02401.nc...
[6/10970] Memproses 2023-01-01 12:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_0500_L2ARP031_FLDK.02401_02401.nc...
[7/10970] Memproses 2023-01-01 13:00:00 WIB...
  -> Mengunduh: NC_H09_20230101_06